- 가상환경 : django_env(python 3.12)

- 프로젝트 명 : django_prj
  
- 앱 이름
    - blog
        - 블로그 기능을 위한 앱
    - single_pages
        - 대문 페이지, 자기소개 페이지를 보여주기 위한 앱
          
- 관리자 계정 생성 후 접속

# 모델 생성

## Post 모델 생성

- 블로그의 포스트의 형태를 정의하는 모델
    - title
        - 게시글의 제목
        - 최대 길이 30
        - 문자열
          
    - content
        - 게시글 내용
        - 길이 제한 없는 텍스트
     
    - created_at
        - 게시글 작성일
        - 날짜 + 시간

In [ ]:
# blog/models.py

class Post(models.Model):
    title = models.CharField(max_length = 30)
    content = models.TextField()
    created_at = models.DateTimeField()

In [ ]:
# blog/admin.py

from blog.models import Post

# Register your models here.
admin.site.register(Post)

## Post 모델 개선

- 관리자 페이지에서 포스트의 제목이 나타나지 않아 글 내용이 어떤 내용인지 파악하기 힘듦
    - "[게시글의id]게시글제목" 형태로 나타나게 수정
        - 예) [1]첫번째 게시글
     
- 서울 기준으로 서버 시간 설정

- 게시글 작성 시각과 수정시각 저장
    - created_at : 작성 시각이 자동으로 입력
 
    - updated_at : 게시글 수정 시각이 자동으로 입력
        - auto_now : 마지막으로 저장된 시점(수정된 시점)을 자동으로 저장

In [ ]:
# blog/models.py

class Post(models.Model):
    title = models.CharField(max_length = 30)
    content = models.TextField()
    created_at = models.DateTimeField(auto_now_add = True)
    updated_at = models.DateTimeField(auto_now = True)

    def __str__(self):
        return f"[{self.pk}]{self.title}"

In [ ]:
# django_prj/settings.py

LANGUAGE_CODE = "en-us"

TIME_ZONE = "Asia/Seoul"

USE_I18N = True

USE_TZ = True

# 웹 페이지 구성

## 필요한 페이지

- 대문 페이지
    - 도메인 또는 서버 IP 주소 뒤에 아무것도 쓰지 않았을 경우, 방문자를 맞이하는 페이지
    - 별도의 하위 페이지는 필요하지 않음
 
- 블로그 페이지
    - blog 접두어를 통해 blog 앱과 연결
      
    - 포스트 목록 페이지
        - URL : 도메인/blog/
        - 블로그의 여러 포스트를 목록 형태로 보여주는 포스트 목록 페이지
        - View : index
        - Template : blog/templates/blog/index.html
     
    - 포스트 상세 페이지
        - URL : 도메인/blog/포스트의pk/
     
- 자기소개 페이지
    - URL : 도메인/about_me/

### 포스트 목록 페이지

In [ ]:
# django_prj/urls.py

from django.urls import path, include

urlpatterns = [
    path("admin/", admin.site.urls),
    path("blog/", include("blog.urls")),
]

In [ ]:
# blog/urls.py

from django.urls import path
from blog import views

urlpatterns = [
    path("", views.index),
]

In [ ]:
# blog/views.py

from blog.models import Post

# Create your views here.
def index(request):
    posts = Post.objects.all()
    context = {
        "posts": posts
    }
    return render(request, "blog/index.html", context)

In [ ]:
# blog/templates/blog/index.html

<!DOCTYPE html>
<html lang="ko">
    <head>
        <title>Blog</title>
    </head>
    <body>
        <h1>Blog</h1>

        {% for p in posts %}
            <hr>
            <h2>{{ p.title }}</h2>
            <h4>{{ p.created_at }}</h4>
            <p>{{ p.content }}</p>
        {% endfor %}
    </body>
</html>

#### 최신 게시글부터 보여주기

- 데이터를 정렬하려면 order_by 를 사용

In [ ]:
# blog/views.py

def index(request):
    posts = Post.objects.all().order_by("-pk")
    context = {
        "posts": posts
    }
    return render(request, "blog/index.html", context)

### 게시글 상세 페이지 생성

- URL
    - 도메인/blog/게시글id/
 
- View
    - single_post_page 함수
    - pk 매개변수로 게시글의 id를 전달받음
 
- Template
    - blog/templates/blog/single_post_page.html
    - 하나의 게시글의 제목, 작성일, 내용 이 나타나게
    - 탭 제목 : "게시글제목 - Blog"

In [ ]:
# blog/urls.py

urlpatterns = [
    path("", views.index),
    path("<int:pk>/", views.single_post_page),
]

In [ ]:
# blog/views.py

def single_post_page(request, pk):
    post = Post.objects.get(pk = pk)
    context = {
        "post": post,
    }
    return render(request, "blog/single_post_page.html", context)

In [ ]:
# blog/templates/blog/single_post_page.html

<!DOCTYPE html>
<html lang="ko">
    <head>
        <title>{{ post.title }} - Blog</title>
    </head>
    <body>
        <nav>
            <a href="/blog/">Blog</a>
        </nav>
        <h1>{{ post.title }}</h1>
        <h4>{{ post.created_at }}</h4>
        <p>{{ post.content }}</p>
        <hr>
    </body>
</html>

## 게시글 제목에 링크 생성

### get_absolute_url() 함수 정의

In [ ]:
# blog/models.py

class Post(models.Model):
    title = models.CharField(max_length = 30)
    content = models.TextField()
    created_at = models.DateTimeField(auto_now_add = True)
    updated_at = models.DateTimeField(auto_now = True)

    def __str__(self):
        return f"[{self.pk}]{self.title}"
    
    def get_absolute_url(self):
        return f"/blog/{self.pk}/"

In [ ]:
# blog/index.html

        {% for p in posts %}
            <hr>
            <h2><a href="{{ p.get_absolute_url }}">{{ p.title }}</a></h2>
            <h4>{{ p.created_at }}</h4>
            <p>{{ p.content }}</p>
        {% endfor %}

## 대문 페이지와 자기소개 페이지 작성

- single_pages 앱의 대문 페이지와 자기소개 페이지 작성

- 대문 페이지
    - 도메인 뒤에 아무것도 붙어 있지 않은 경우에는 single_pages 앱에서 처리
    - URL : 도메인/
    - View : landing 함수에서 templates/single_pages/landing.html 렌더링
    - 템플릿 : 프론트엔드 기초의 index.html 내용 인용
 
- 자기소개 페이지
    - URL : 도메인/about_me/
    - View : about_me 함수에서 templates/single_pages/about_me.html
    - 템플릿 : 프론트엔드 기초의 about_me.html 내용 인용

In [ ]:
# django_prj/urls.py

urlpatterns = [
    path("admin/", admin.site.urls),
    path("blog/", include("blog.urls")),
    path("", include("single_pages.urls")),
]

In [ ]:
# single_pages/urls.py

from django.urls import path
from single_pages import views

urlpatterns = [
    path("", views.landing),
    path("about_me/", views.about_me),
]

In [ ]:
# single_pages/views.py

def landing(request):
    return render(request, "single_pages/landing.html")

def about_me(request):
    return render(request, "single_pages/about_me.html")

# views 작성

- views.py 를 작성하는 방법은 2가지가 있음
    - FBV(Function based view)
        - 함수에 기반을 둔 방법
        - 함수를 직접 만들어서 원하는 기능을 직접 구현
 
    - CBV(Class based view)
        - 장고가 제공하는 클래스를 활용해 구현하는 방법
     
- FBV와 CBV 중 어느 것이 더 낫다고 단정지을 수는 없고 필요에 따라, 취향에 따라 선택하면 됨

## CBV로 페이지 작성

### ListView 로 게시글 목록 페이지 작성

- FBV의 index() 함수를 대체하는 PostList 클래스를 ListView 클래스를 상속해서 작성

In [ ]:
# blog/views.py

# index 함수를 대체하는 class
class PostList(ListView):
    model = Post

In [ ]:
# blog/urls.py

urlpatterns = [
    # path("", views.index),
    path("", views.PostList.as_view()),
    path("<int:pk>/", views.single_post_page),
]

#### 템플릿 파일 지정

- 포스트 목록 URL 로 접속하면 post_list.html 이 필요하다고 함
    - 장고가 제공하는 ListView 는 모델명 뒤에 "_list" 가 붙은 html 파일을 기본 템플릿으로 사용함
        - 즉, Post 모델을 사용하려면 post_list.html 이 필요
            - 해결방법
                1. PostList 클래스에서 template_name 을 직접 지정
                2. post_list.html 을 만들기

- template_name 을 지정하는 방법

In [ ]:
# blog/views.py

# index 함수를 대체하는 class
class PostList(ListView):
    model = Post
    template_name = "blog/index.html"

- FBV에서는 딕셔너리로 데이터를 전달하지만, ListView 클래스에서는 "모델명_list" 또는 "object_list" 라는 이름으로 전달함

In [ ]:
# blog/index.html

        {% for p in post_list %}
            <hr>
            <h2><a href="{{ p.get_absolute_url }}">{{ p.title }}</a></h2>
            <h4>{{ p.created_at }}</h4>
            <p>{{ p.content }}</p>
        {% endfor %}

- post_list.html 을 만들어서 해결하는 방법
    - views 의 template_name 을 삭제 후
    - blog/templates/blog/index.html 의 파일명을 post_list.html 로 수정

#### 최신 포스트부터 보여주기

- ListView의 ordering 변수는 정렬 기능을 구현

In [ ]:
# blog/views.py

# index 함수를 대체하는 class
class PostList(ListView):
    model = Post
    # template_name = "blog/index.html"
    ordering = "-pk"

### CBV로 포스트 상세 페이지 작성

#### DetailView로 포스트 상세 페이지 작성

- 한 데이터를 상세보기 할 때는 DetailView 를 이용

In [ ]:
# blog/views.py

class PostDetail(DetailView):
    model = Post

In [ ]:
# blog/urls.py

urlpatterns = [
    path("", views.PostList.as_view()),
    path("<int:pk>/", views.PostDetail.as_view()),
]

- 템플릿 파일 지정
    - PostDetail 클래스에서 template_name 을 수정해 해결할 수도 있고
    - single_post_page.html 의 파일명을 post_detail.html 로 수정할 수도 있음